In [10]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from sklearn.model_selection import train_test_split
import numpy as np


df = pd.read_csv('imdb.csv')
labels = df['sentiment'].map({'positive': 1, 'negative': 0}).values
texts = df['review'].values


VOCAB_SIZE = 10000
MAX_LEN = 200
EMBEDDING_DIM = 64


tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post')
y = np.array(labels)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)


def initialize_model():
    model = Sequential([
        Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LEN),
        LSTM(64, return_sequences=True),
        LSTM(32),
        Dense(24, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


model = initialize_model()
model.summary()

history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test, y_test),
    verbose=1
)

c:\Users\darla\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 63s 104ms/step - accuracy: 0.5967 - loss: 0.6446 - val_accuracy: 0.7624 - val_loss: 0.5490
Epoch 2/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 52s 89ms/step - accuracy: 0.5930 - loss: 0.6577 - val_accuracy: 0.6224 - val_loss: 0.6557
Epoch 3/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 52s 88ms/step - accuracy: 0.7681 - loss: 0.4865 - val_accuracy: 0.7981 - val_loss: 0.4368
Epoch 4/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 52s 89ms/step - accuracy: 0.8890 - loss: 0.2804 - val_accuracy: 0.8882 - val_loss: 0.2733
Epoch 5/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 52s 89ms/step - accuracy: 0.9226 - loss: 0.2095 - val_accuracy: 0.8932 - val_loss: 0.2580


In [11]:
def predict_sentiment(text):
    sequence = tokenizer.texts_to_sequences([text])
    padded_sequence = pad_sequences(sequence, maxlen=MAX_LEN, padding='post')
    prediction = model.predict(padded_sequence)
    sentiment = "positif" if prediction[0][0] > 0.5 else "négatif"
    confidence = prediction[0][0] if sentiment == "positif" else 1 - prediction[0][0]
    return sentiment, confidence

new_review = "This movie was fantastic! The actors were brilliant and the plot was engaging."
sentiment, confidence = predict_sentiment(new_review)
print(f"Sentiment: {sentiment}, Confiance: {confidence:.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 223ms/step
Sentiment: positif, Confiance: 0.94
